# Projeto VISÃO — Tarefa 1.5 no Colab (psMNIST completo)
Reconstrói o harness `visao/bench/timeseries.py` e roda a comparação CfC vs LSTM
em **psMNIST completo** (60k treino / 10k teste), 50-100 épocas, em GPU.
Objetivo: validar a alegação 'empatar ou superar LSTM/GRU com ~10x menos parâmetros'.

Fluxo:
1. Upload dos 4 arquivos MNIST (ou baixar do mirror).
2. Instalar jax[cuda] ou usar jax[cpu] (T4 free tem GPU).
3. Rodar célula de benchmark.
4. `results_smnist_full.json` é gerado -> baixar e trazer para o repo.

In [ ]:
import os, gzip, json, time
import numpy as np
import jax
import jax.numpy as jnp
print('jax', jax.__version__, 'devices:', jax.devices())

In [ ]:
# --- dados: upload dos 4 arquivos IDX ou baixar do mirror ---
DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)

files = {
  'train-images-idx3-ubyte.gz': 'https://ossci-datasets.s3.amazonaws.com/mnist/train-images-idx3-ubyte.gz',
  'train-labels-idx1-ubyte.gz': 'https://ossci-datasets.s3.amazonaws.com/mnist/train-labels-idx1-ubyte.gz',
  't10k-images-idx3-ubyte.gz':   'https://ossci-datasets.s3.amazonaws.com/mnist/t10k-images-idx3-ubyte.gz',
  't10k-labels-idx1-ubyte.gz':   'https://ossci-datasets.s3.amazonaws.com/mnist/t10k-labels-idx1-ubyte.gz',
}
# Se já upou os arquivos manualmente, descomente e aponte para /content:
# DATA_DIR = '/content'
import urllib.request
for fn, url in files.items():
    dst = os.path.join(DATA_DIR, fn)
    if not os.path.exists(dst):
        print('baixando', fn)
        urllib.request.urlretrieve(url, dst)
print('dados prontos em', DATA_DIR)

In [ ]:
# --- harness (reconstrói visao/bench/timeseries.py) ---
PERM = {}
def _idx_reader(path, n_items, rows, cols, header):
    with gzip.open(path, 'rb') as f:
        f.read(header)
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data.reshape(n_items, rows, cols)

def load_smnist_permuted(n_train=None, n_test=None, perm_seed=42, data_dir=DATA_DIR):
    if perm_seed not in PERM:
        rng = np.random.default_rng(perm_seed)
        PERM[perm_seed] = rng.permutation(784)
    perm = PERM[perm_seed]
    x_tr = _idx_reader(os.path.join(data_dir,'train-images-idx3-ubyte.gz'),60000,28,28,16).reshape(60000,784)/255.0
    y_tr = _idx_reader(os.path.join(data_dir,'train-labels-idx1-ubyte.gz'),60000,1,1,8).reshape(60000).astype(np.int32)
    x_te = _idx_reader(os.path.join(data_dir,'t10k-images-idx3-ubyte.gz'),10000,28,28,16).reshape(10000,784)/255.0
    y_te = _idx_reader(os.path.join(data_dir,'t10k-labels-idx1-ubyte.gz'),10000,1,1,8).reshape(10000).astype(np.int32)
    x_tr = x_tr[:, perm].reshape(-1,784,1).astype(np.float32)
    x_te = x_te[:, perm].reshape(-1,784,1).astype(np.float32)
    if n_train is not None: x_tr, y_tr = x_tr[:n_train], y_tr[:n_train]
    if n_test is not None:   x_te, y_te = x_te[:n_test], y_te[:n_test]
    return x_tr, y_tr, x_te, y_te

def adam_init(params):
    return {'mu': jax.tree_util.tree_map(jnp.zeros_like, params),
            'nu': jax.tree_util.tree_map(jnp.zeros_like, params), 't': 0}
def adam_update(params, grads, opt, lr=1e-3, b1=0.9, b2=0.999, eps=1e-8):
    t = opt['t'] + 1
    mu = jax.tree_util.tree_map(lambda m,g: b1*m+(1-b1)*g, opt['mu'], grads)
    nu = jax.tree_util.tree_map(lambda v,g: b2*v+(1-b2)*(g*g), opt['nu'], grads)
    mhat = jax.tree_util.tree_map(lambda m: m/(1-b1**t), mu)
    nhat = jax.tree_util.tree_map(lambda v: v/(1-b2**t), nu)
    new = jax.tree_util.tree_map(lambda p,mh,nh: p - lr*mh/(jnp.sqrt(nh)+eps), params, mhat, nhat)
    return new, {'mu':mu,'nu':nu,'t':t}

def cfc_params(cfc_hidden=64, sparsity=0.5, seed=0, tau_min=0.5, tau_max=50.0):
    rng = np.random.default_rng(seed); h = cfc_hidden
    scale_in = 1.0/np.sqrt(1); scale_rec = 1.0/np.sqrt(h)
    W_in = rng.normal(0, scale_in, (h,1)).astype(np.float32)
    W_rec = rng.normal(0, scale_rec, (h,h)).astype(np.float32)
    mask = (rng.random((h,h)) > sparsity).astype(np.float32); np.fill_diagonal(mask, 0.0)
    W_rec = W_rec * mask
    b = np.zeros(h, dtype=np.float32); A = rng.normal(0,1.0,h).astype(np.float32)
    tau = np.exp(rng.uniform(np.log(tau_min), np.log(tau_max), h))
    tau_raw = np.log(np.exp(tau)-1.0).astype(np.float32)
    W_out = rng.normal(0, 1.0/np.sqrt(h), (10,h)).astype(np.float32)
    b_out = np.zeros(10, dtype=np.float32)
    return {'W_in':jnp.asarray(W_in),'W_rec':jnp.asarray(W_rec),'b':jnp.asarray(b),
            'A':jnp.asarray(A),'tau_raw':jnp.asarray(tau_raw),'W_out':jnp.asarray(W_out),
            'b_out':jnp.asarray(b_out),'mask':jnp.asarray(mask)}

def make_cfc_forward(dt=0.1):
    def forward(params, seq):
        h = jnp.zeros(params['W_in'].shape[0]); tau = jax.nn.softplus(params['tau_raw'])+1e-3; mask = params['mask']
        def body(h, x):
            f = jax.nn.sigmoid(params['W_in'] @ x + (params['W_rec']*mask) @ h + params['b'])
            h = (h + dt*(f*params['A'])) / (1.0 + dt*(1.0/tau + f))
            return h, h
        _, states = jax.lax.scan(body, h, seq)
        return params['W_out'] @ jnp.mean(states, axis=0) + params['b_out']
    return forward

def lstm_params(lstm_hidden=128, seed=1):
    rng = np.random.default_rng(seed); h = lstm_hidden; s = 1.0/np.sqrt(h)
    W = rng.normal(0, s, (4*h, h+1)).astype(np.float32); b = np.zeros(4*h, dtype=np.float32)
    W_out = rng.normal(0, 1.0/np.sqrt(h), (10,h)).astype(np.float32); b_out = np.zeros(10, dtype=np.float32)
    return {'W':jnp.asarray(W),'b':jnp.asarray(b),'W_out':jnp.asarray(W_out),'b_out':jnp.asarray(b_out)}
def lstm_forward(params, seq):
    h = jnp.zeros(params['W'].shape[0]//4); c = jnp.zeros_like(h); W, b = params['W'], params['b']
    def body(carry, x):
        h,c = carry; cat = jnp.concatenate([h,x]); g = W@cat + b
        hh = h.shape[0]
        i = jax.nn.sigmoid(g[:hh]); f = jax.nn.sigmoid(g[hh:2*hh]); o = jax.nn.sigmoid(g[2*hh:3*hh]); gc = jnp.tanh(g[3*hh:])
        c = f*c + i*gc; h = o*jnp.tanh(c); return (h,c), None
    (h,_),_ = jax.lax.scan(body, (h,c), seq)
    return params['W_out'] @ h + params['b_out']

def count_params(params):
    return int(sum(int(p.size) for k,p in params.items() if k != 'mask'))
def _xent(logits, y):
    onehot = jax.nn.one_hot(y, 10)
    logp = logits - jax.scipy.special.logsumexp(logits, axis=-1, keepdims=True)
    return -jnp.mean(logp * onehot)
def _train(forward, params_init, x, y, n_epochs, batch, lr):
    params = params_init; opt = adam_init(params); n = x.shape[0]; rng = np.random.default_rng(123)
    grad_fn = jax.jit(jax.grad(lambda p,xb,yb: _xent(jax.vmap(forward, in_axes=(None,0))(p,xb), yb)))
    fwd_jit = jax.jit(lambda p,xb: jax.vmap(forward, in_axes=(None,0))(p,xb))
    for _ in range(n_epochs):
        order = rng.permutation(n)
        for i in range(0, n, batch):
            idx = order[i:i+batch]
            g = grad_fn(params, x[idx], y[idx]); params, opt = adam_update(params, g, opt, lr=lr)
    return params, fwd_jit
def _accuracy(forward_jit, params, x, y, batch=256):
    correct = 0; n = x.shape[0]
    for i in range(0, n, batch):
        logits = forward_jit(params, x[i:i+batch]); correct += int(jnp.sum(jnp.argmax(logits, axis=-1) == y[i:i+batch]))
    return correct/n
print('harness pronto')

In [ ]:
# --- benchmark psMNIST COMPLETO ---
N_EPOCHS = 50          # suba para 100 se quiser curva mais estável
BATCH = 128
LR = 1e-3
CFC_H = 64
LSTM_H = 128

x_tr, y_tr, x_te, y_te = load_smnist_permuted(n_train=None, n_test=None, perm_seed=42, data_dir=DATA_DIR)
y_tr = jnp.asarray(y_tr); y_te = jnp.asarray(y_te)
print('train', x_tr.shape, 'test', x_te.shape)

t0 = time.time()
cfc_p = cfc_params(cfc_hidden=CFC_H, seed=0)
cfc_fwd = make_cfc_forward()
cfc_p, cfc_jit = _train(cfc_fwd, cfc_p, x_tr, y_tr, N_EPOCHS, BATCH, LR)
cfc_acc = _accuracy(cfc_jit, cfc_p, x_te, y_te)
cfc_params = count_params(cfc_p)
print(f'CfC  acc={cfc_acc:.4f}  params={cfc_params}  ({time.time()-t0:.0f}s)')

t0 = time.time()
lstm_p = lstm_params(lstm_hidden=LSTM_H, seed=1)
lstm_p, lstm_jit = _train(lstm_forward, lstm_p, x_tr, y_tr, N_EPOCHS, BATCH, LR)
lstm_acc = _accuracy(lstm_jit, lstm_p, x_te, y_te)
lstm_params = count_params(lstm_p)
print(f'LSTM acc={lstm_acc:.4f}  params={lstm_params}  ({time.time()-t0:.0f}s)')

result = {
  'config': {'n_epochs': N_EPOCHS, 'batch': BATCH, 'lr': LR, 'cfc_hidden': CFC_H, 'lstm_hidden': LSTM_H,
            'dataset': 'psMNIST full 60k/10k', 'device': str(jax.devices())},
  'cfc': {'accuracy': float(cfc_acc), 'params': cfc_params},
  'lstm': {'accuracy': float(lstm_acc), 'params': lstm_params},
  'param_ratio_lstm_over_cfc': float(lstm_params / cfc_params),
  'claim_10x_less_params': bool(lstm_params / cfc_params >= 8.0),
  'cfc_beats_or_ties_lstm': bool(cfc_acc >= lstm_acc - 0.02),
}
with open('results_smnist_full.json', 'w') as f:
    json.dump(result, f, indent=2)
print(json.dumps(result, indent=2))

In [ ]:
# baixar o JSON de resultado
from google.colab import files
files.download('results_smnist_full.json')